# Workforce Performance & Productivity Analytics
### An exploratory data analysis of employee outcomes, compensation, workload, and retention

**Project type:** Workforce / HR analytics · **Dataset:** 100,000 employee records · **Tools:** Python, pandas, NumPy, Matplotlib, Seaborn, SciPy, scikit-learn

---

## Executive overview

This project investigates how employee performance scores relate to compensation, workload, professional development, satisfaction, and resignation status. The goal is to move beyond descriptive summaries and identify patterns that could inform workforce planning.

**Questions explored**
1. How are performance scores distributed across the workforce?
2. Do performance and compensation vary across departments or job roles?
3. Which numerical factors have the strongest observed relationships with performance?
4. How do resignation status and employee characteristics compare?
5. What can statistical testing and feature-association analysis tell us—and what can they not tell us?

> **Important:** These are observational data. Associations do not establish causation. The dataset should not be used to make individual employment decisions.

**Current scope:** This release is an exploratory analytics study. Predictive modeling is a planned extension; no model performance is claimed in this notebook.


## 1. Environment and reproducibility

The notebook uses commonly available Python libraries and avoids notebook-specific installation commands. If a package is missing in your environment, install it separately with `pip`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr, kruskal
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OneHotEncoder

SEED = 42
np.random.seed(SEED)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "figure.dpi": 110,
})

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 2. Load the data

The loader checks a local project folder first, then a Kaggle input path. Place the CSV beside this notebook when running the project locally.

In [ ]:
DATA_CANDIDATES = [
    Path("Extended_Employee_Performance_and_Productivity_Data.csv"),
    Path("data/Extended_Employee_Performance_and_Productivity_Data.csv"),
    Path("/kaggle/input/datasets/hamidabou/employeeperformance/Extended_Employee_Performance_and_Productivity_Data.csv"),
]

data_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Dataset not found. Place the CSV beside this notebook or update DATA_CANDIDATES."
    )

df = pd.read_csv(data_path)
print(f"Loaded: {data_path}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]:,}")
display(df.head())

## 3. Data quality and structure

Before interpreting patterns, inspect the schema, missingness, duplicate records, and basic validity checks.

In [ ]:
print("Dataset dimensions:", df.shape)
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=False)
}).sort_values(["missing_count", "unique_values"], ascending=[False, False]))

print(f"\nExact duplicate rows: {df.duplicated().sum():,}")
print(f"Duplicate employee IDs: {df['Employee_ID'].duplicated().sum():,}")
display(df.describe(include="all").T)

In [ ]:
# Domain-oriented validation checks
quality_checks = pd.Series({
    "Performance score outside 1–5": (~df["Performance_Score"].between(1, 5)).sum(),
    "Age outside 18–100": (~df["Age"].between(18, 100)).sum(),
    "Negative monthly salary": (df["Monthly_Salary"] < 0).sum(),
    "Negative weekly work hours": (df["Work_Hours_Per_Week"] < 0).sum(),
    "Remote frequency outside 0–100": (~df["Remote_Work_Frequency"].between(0, 100)).sum(),
    "Negative tenure": (df["Years_At_Company"] < 0).sum(),
})
display(quality_checks.rename("records_flagged").to_frame())

## 4. Workforce profile

Establish the composition of the dataset before comparing outcomes. Category counts provide context for interpreting department- and role-level results.

In [ ]:
categorical_cols = ["Department", "Job_Title", "Education_Level", "Gender"]
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for col, ax in zip(categorical_cols, axes.flat):
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, ax=ax, color=sns.color_palette("deep")[0])
    ax.set_title(f"Employee count by {col.replace('_', ' ')}")
    ax.set_xlabel("Number of employees")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 5. Performance score distribution

The performance score is treated as an ordered outcome (1 = lower score, 5 = higher score). Check both the count and percentage distribution.

In [ ]:
score_summary = (
    df["Performance_Score"].value_counts().sort_index()
    .rename_axis("Performance_Score").to_frame("Employees")
)
score_summary["Share_pct"] = score_summary["Employees"] / len(df) * 100
display(score_summary.round(2))

ax = sns.countplot(data=df, x="Performance_Score", color=sns.color_palette("deep")[1])
ax.set(title="Distribution of employee performance scores",
       xlabel="Performance score", ylabel="Number of employees")
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f")
plt.tight_layout()
plt.show()

## 6. Performance across departments and job roles

Compare medians and distributions—not just averages. Differences between groups may reflect role mix, measurement practices, or other unobserved factors.

In [ ]:
department_perf = (
    df.groupby("Department")["Performance_Score"]
      .agg(Employees="size", Mean="mean", Median="median", Std="std")
      .sort_values("Mean", ascending=False)
)
display(department_perf.round(2))

plt.figure(figsize=(12, 6))
order = department_perf.index
sns.boxplot(data=df, x="Department", y="Performance_Score", order=order)
plt.title("Performance score by department")
plt.xlabel("")
plt.ylabel("Performance score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

role_perf = (
    df.groupby("Job_Title")["Performance_Score"]
      .agg(Employees="size", Mean="mean", Median="median")
      .sort_values("Mean", ascending=False)
)
display(role_perf.round(2))

plt.figure(figsize=(11, 5))
sns.boxplot(data=df, x="Job_Title", y="Performance_Score",
            order=role_perf.index)
plt.title("Performance score by job title")
plt.xlabel("")
plt.ylabel("Performance score")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 7. Compensation and performance

Explore whether salary tends to increase across performance-score groups. Use distribution plots and robust rank-based statistics alongside correlation.

In [ ]:
salary_by_score = (
    df.groupby("Performance_Score")["Monthly_Salary"]
      .agg(Employees="size", Mean="mean", Median="median", Std="std",
           Min="min", Max="max")
)
display(salary_by_score.round(2))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(data=df, x="Performance_Score", y="Monthly_Salary",
            ax=axes[0], color=sns.color_palette("deep")[2])
axes[0].set_title("Monthly salary by performance score")
axes[0].set_xlabel("Performance score")
axes[0].set_ylabel("Monthly salary")

sns.regplot(data=df, x="Monthly_Salary", y="Performance_Score",
            scatter_kws={"alpha": 0.06, "s": 10},
            line_kws={"color": "black"}, ax=axes[1])
axes[1].set_title("Salary and performance: overall trend")
axes[1].set_xlabel("Monthly salary")
axes[1].set_ylabel("Performance score")
plt.tight_layout()
plt.show()

## 8. Workload, development, and satisfaction

Examine the relationship between performance and selected employee-level measures. Correlation is useful for screening patterns, but near-zero correlation does not rule out nonlinear or subgroup relationships.

In [ ]:
numeric_features = [
    "Age", "Years_At_Company", "Monthly_Salary",
    "Work_Hours_Per_Week", "Projects_Handled", "Overtime_Hours",
    "Sick_Days", "Remote_Work_Frequency", "Team_Size",
    "Training_Hours", "Promotions", "Employee_Satisfaction_Score",
    "Performance_Score"
]
corr = df[numeric_features].corr(numeric_only=True)

plt.figure(figsize=(13, 10))
sns.heatmap(corr, cmap="vlag", center=0, annot=True, fmt=".2f",
            linewidths=.4, square=True)
plt.title("Correlation matrix of numerical workforce variables")
plt.tight_layout()
plt.show()

performance_corr = (
    corr["Performance_Score"].drop("Performance_Score")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)
display(performance_corr.rename("Pearson_correlation").to_frame())

## 9. Retention and resignation patterns

Compare resignation rates and workforce characteristics by resignation status. These are descriptive differences, not evidence that any variable causes an employee to resign.

In [ ]:
retention_summary = (
    df.groupby("Resigned")
      .agg(Employees=("Employee_ID", "size"),
           Mean_Performance=("Performance_Score", "mean"),
           Mean_Salary=("Monthly_Salary", "mean"),
           Mean_Satisfaction=("Employee_Satisfaction_Score", "mean"),
           Mean_Tenure=("Years_At_Company", "mean"))
)
retention_summary["Workforce_share_pct"] = retention_summary["Employees"] / len(df) * 100
display(retention_summary.round(2))

resignation_by_department = (
    df.groupby("Department")["Resigned"].mean()
      .mul(100).sort_values(ascending=False)
      .rename("Resignation_rate_pct").to_frame()
)
display(resignation_by_department.round(2))

plt.figure(figsize=(10, 5))
sns.barplot(x=resignation_by_department.index,
            y=resignation_by_department["Resignation_rate_pct"],
            color=sns.color_palette("deep")[3])
plt.title("Resignation rate by department")
plt.xlabel("")
plt.ylabel("Resigned employees (%)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 10. Hire-date and tenure context

Parse hire dates and inspect the observed hiring timeline. The date field may be useful for cohort analysis, but it should not be interpreted as a complete employment history without additional context.

In [ ]:
df["Hire_Date"] = pd.to_datetime(df["Hire_Date"], errors="coerce")
print("Hire date range:", df["Hire_Date"].min(), "to", df["Hire_Date"].max())
print("Unparseable hire dates:", df["Hire_Date"].isna().sum())

hires_by_month = (
    df.dropna(subset=["Hire_Date"])
      .assign(Hire_Month=lambda x: x["Hire_Date"].dt.to_period("M").dt.to_timestamp())
      .groupby("Hire_Month").size()
)
plt.figure(figsize=(13, 5))
hires_by_month.plot()
plt.title("Employee hires represented in the dataset by month")
plt.xlabel("Hire month")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

## 11. Statistical testing: salary and performance

Use Spearman rank correlation to assess monotonic association and the Kruskal–Wallis test to compare salary distributions across performance groups. With a large sample, small differences can be statistically significant; interpret effect magnitude and practical relevance.

In [ ]:
salary = df["Monthly_Salary"]
score = df["Performance_Score"]

spearman = spearmanr(salary, score)
groups = [g["Monthly_Salary"].to_numpy()
          for _, g in df.groupby("Performance_Score")]
kw = kruskal(*groups)

# Epsilon-squared is a simple effect-size estimate for Kruskal–Wallis.
n = len(df)
k = len(groups)
epsilon_sq = max(0, (kw.statistic - k + 1) / (n - k))

print(f"Spearman rho: {spearman.statistic:.4f}")
print(f"Spearman p-value: {spearman.pvalue:.3e}")
print(f"Kruskal–Wallis H: {kw.statistic:.3f}")
print(f"Kruskal–Wallis p-value: {kw.pvalue:.3e}")
print(f"Approx. epsilon-squared: {epsilon_sq:.4f}")

## 12. Feature association with performance

Estimate mutual information between the performance score and employee attributes.
Categorical variables are one-hot encoded and those binary indicators are explicitly
marked as discrete for the estimator.

`Resigned` is excluded from this analysis because resignation may occur after the
performance measurement or be related to subsequent employment events. Including it
without a verified timeline could introduce information leakage.

Mutual information captures statistical dependence, including some nonlinear
relationships. It is not causal importance and does not establish predictive usefulness
on unseen data.


In [ ]:
target = df["Performance_Score"].astype(int)

# Exclude identifiers, target, date, and potentially post-outcome resignation status.
features = df.drop(
    columns=["Employee_ID", "Performance_Score", "Hire_Date", "Resigned"]
).copy()

cat_cols = features.select_dtypes(include=["object", "bool"]).columns.tolist()
num_cols = features.select_dtypes(include=np.number).columns.tolist()

# One-hot encode categorical features.
X_cat = pd.get_dummies(
    features[cat_cols].astype("category"),
    drop_first=False,
    dtype=float
)
X_num = features[num_cols].astype(float)
X_mi = pd.concat([X_num, X_cat], axis=1)

# Numeric columns are treated as continuous; one-hot indicators as discrete.
discrete_mask = np.array(
    [False] * X_num.shape[1] + [True] * X_cat.shape[1]
)

mi = mutual_info_classif(
    X_mi,
    target,
    discrete_features=discrete_mask,
    random_state=SEED
)
mi_scores = pd.Series(
    mi, index=X_mi.columns
).sort_values(ascending=False)

display(mi_scores.head(15).rename("Mutual_information").to_frame())

plt.figure(figsize=(10, 7))
mi_scores.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 features by mutual information with performance")
plt.xlabel("Mutual information")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 13. Key findings and interpretation

Use the outputs above to complete this section after running the notebook. Separate observed patterns from hypotheses, and avoid overstating weak associations.

**Interpretation checklist**
- Which findings are large enough to matter operationally?
- Are group sizes sufficiently comparable?
- Could department or job-role differences explain apparent relationships?
- Do statistical significance and practical significance tell the same story?
- What additional data would be needed to evaluate retention or performance interventions?

In [ ]:
# Compact, reproducible headline metrics
headline = {
    "Employees": len(df),
    "Departments": df["Department"].nunique(),
    "Job titles": df["Job_Title"].nunique(),
    "Missing cells": int(df.isna().sum().sum()),
    "Duplicate rows": int(df.duplicated().sum()),
    "Mean performance score": df["Performance_Score"].mean(),
    "Mean monthly salary": df["Monthly_Salary"].mean(),
    "Resignation rate (%)": df["Resigned"].mean() * 100,
    "Salary-performance Pearson r": df["Monthly_Salary"].corr(df["Performance_Score"]),
}
display(pd.Series(headline, name="Value").to_frame().round(3))

## 14. Limitations and next steps

### Limitations
- The analysis is observational and cannot establish causal effects.
- The dataset's generation process and representativeness are not established in this notebook.
- Performance scores may reflect subjective or organization-specific evaluation practices.
- Aggregated group differences can conceal variation within teams and roles.
- Statistical tests can produce very small p-values in large samples, even for modest effects.
- The resignation indicator does not provide timing, reasons for departure, or a full employee lifecycle.

### Recommended extensions
1. **Cohort analysis:** compare hiring cohorts and tenure groups.
2. **Fairness audit:** assess score and resignation patterns across demographic groups, with careful attention to legal and ethical constraints.
3. **Predictive modeling:** only after establishing a meaningful prediction target, leakage-safe split strategy, baseline, and evaluation plan.
4. **Explainability:** compare model explanations with domain knowledge and validate stability across groups.
5. **Operational dashboard:** communicate a small set of validated KPIs with clear definitions and caveats.

## Conclusion

This notebook provides a reproducible foundation for workforce analytics by combining data-quality checks, descriptive exploration, group comparisons, statistical testing, and feature-association analysis. The strongest portfolio contribution is not simply producing charts—it is demonstrating disciplined interpretation, reproducibility, and awareness of the limits of observational data.

## Reproducibility

- Random seed: `42`
- Dataset file: `Extended_Employee_Performance_and_Productivity_Data.csv`
- Run order: execute all cells from top to bottom.
- Suggested repository layout:

```text
employee-performance-analytics/
├── Performance.ipynb
├── Extended_Employee_Performance_and_Productivity_Data.csv
└── README.md
```

If the dataset license does not permit redistribution, do not commit the CSV. Instead, document the original dataset source and instructions for obtaining it.